# Load trained `scDiffEq` model

This example highlights loading a model that was trained on the full LARRY dataset for fate perturbation (**Task 3** in the scDiffEq manuscript).

### Import dependencies

In [2]:
import larry
import scdiffeq as sdq
import scdiffeq_analyses as sdq_an

F_obs = larry.tasks.fate_prediction.F_obs

### Load data  
加载数据 因为代码内置使用figshare 可以先离线下载  直接加载

In [2]:
adata = sdq.datasets.larry(data_dir="/data/cai803/scDiffEq/data", force_download=False)
adata

AnnData object with n_obs × n_vars = 130887 × 2447
    obs: 'Library', 'Cell barcode', 'Time point', 'Starting population', 'Cell type annotation', 'Well', 'SPRING-x', 'SPRING-y', 'clone_idx', 'fate_observed', 't0_fated'
    var: 'gene_ids', 'hv_genes', 'use_genes'
    uns: 'fate_counts', 'time_occupance'
    obsm: 'X_clone', 'X_pca', 'X_scaled', 'cell_fate_df'

### Load the project (`sdq.io.Project`)  模型加载流程 (核心)

**Important**: first, clone and locally install the `scdiffeq-analyses` repo:

```shell
git clone https://github.com/scDiffEq/scdiffeq-analyses.git;
cd scdiffeq-analyses; pip install -e .
```
下载好了

In [3]:
#权重就在下载的文件夹里面
project_path = "/data/cai803/scDiffEq/scdiffeq-analyses-main/manuscript/models/LARRY.full_dataset/LightningSDE-FixedPotential-RegularizedVelocityRatio/"
project = sdq.io.Project(path=project_path)

### Get the best checkpoints
获取最佳检查点 (best_checkpoints)：

In [4]:
best_ckpts_df = sdq_an.parsers.best_checkpoints(project=project)
best_ckpts_df

,train,test,ckpt_path,epoch
version_0,0.571656,0.551804,/data/cai803/scDiffEq/scdiffeq-analyses-main/m...,2500
version_1,0.541401,0.465658,/data/cai803/scDiffEq/scdiffeq-analyses-main/m...,1706
version_2,0.547771,0.499418,/data/cai803/scDiffEq/scdiffeq-analyses-main/m...,1238
version_3,0.496815,0.504075,/data/cai803/scDiffEq/scdiffeq-analyses-main/m...,1245
version_4,0.562102,0.522701,/data/cai803/scDiffEq/scdiffeq-analyses-main/m...,1662


In [5]:
ckpt_path = best_ckpts_df['ckpt_path'].loc['version_0']

### Load the model  
实例化模型 (sdq.io.load_model)：  
使用 load_model 函数，将之前加载的数据 (adata) 和选中的检查点路径 (ckpt_path) 结合，重建了完整的 scDiffEq 模型对象。

In [6]:
model = sdq.io.load_model(adata = adata, ckpt_path = ckpt_path)
print(model)
model.to("mps:0") # or "cuda:0", for example

Seed set to 0


scDiffEq


### Alternatively: load only the `DiffEq`

In [7]:
DiffEq = sdq.io.load_diffeq(ckpt_path=ckpt_path)
print(DiffEq)

LightningSDE-FixedPotential-RegularizedVelocityRatio
